In [1]:
# RV 2024 dataset quick profile (chunked; safe for ~400k rows)

from pathlib import Path
import math
from collections import Counter, defaultdict

DATA_PATH = Path("/home/canelo/uncertainty_quantification/datasets/rv_dataset_2024.csv")
TARGET = "y_true"

# --- try pandas first (preferred) ---
try:
    import pandas as pd
    import numpy as np

    # columns we expect to treat as categorical (adjust if needed)
    CAT_COLS = ["mbv_code", "mbv_model", "vda_make", "vda_model", "fuel", "latest_sales_quarter"]
    TOPK = 15
    UNIQUE_CAP = 5000   # stop tracking exact uniques beyond this per column

    # running stats
    n_rows = 0
    missing = Counter()
    num_stats = {}  # col -> dict
    cat_top = {c: Counter() for c in CAT_COLS}
    cat_uniques = {c: set() for c in CAT_COLS}
    cat_capped = {c: False for c in CAT_COLS}

    # for a quick leakage/consistency check in RV: y_true ?= 100*market_value/listprice
    max_abs_diff_ratio = -1.0
    n_ratio_checked = 0

    # read in chunks to avoid holding full df in memory
    for chunk in pd.read_csv(DATA_PATH, chunksize=200_000, low_memory=False):
        n_rows += len(chunk)

        # normalize possible unnamed index col
        if "Unnamed: 0" in chunk.columns:
            chunk = chunk.drop(columns=["Unnamed: 0"])

        # missingness (treat NaN only; blank strings for object handled below in cats)
        missing.update(chunk.isna().sum().to_dict())

        # numeric columns: detect by pandas dtype each chunk (coerce object->numeric where possible)
        for col in chunk.columns:
            s = chunk[col]
            if col in CAT_COLS:
                continue

            # try numeric
            s_num = pd.to_numeric(s, errors="coerce")
            ok = s_num.notna()
            if ok.sum() == 0:
                continue

            x = s_num[ok].to_numpy(dtype=float)
            st = num_stats.setdefault(col, {"n": 0, "sum": 0.0, "sum2": 0.0, "min": math.inf, "max": -math.inf})
            st["n"] += int(x.size)
            st["sum"] += float(x.sum())
            st["sum2"] += float((x * x).sum())
            st["min"] = min(st["min"], float(np.nanmin(x)))
            st["max"] = max(st["max"], float(np.nanmax(x)))

        # categorical columns: top values + unique counts (capped)
        for c in CAT_COLS:
            if c not in chunk.columns:
                continue
            s = chunk[c].astype("string")
            # count blank/empty as missing too
            missing[c] += int((s.isna() | (s.str.strip() == "")).sum())
            s = s.fillna("").str.strip()
            s = s[s != ""]
            cat_top[c].update(s.value_counts().to_dict())

            if not cat_capped[c]:
                # track uniques until cap
                for v in s.unique():
                    cat_uniques[c].add(str(v))
                    if len(cat_uniques[c]) >= UNIQUE_CAP:
                        cat_capped[c] = True
                        break

        # consistency check (sample within each chunk)
        if {"market_value", "listprice_with_equipment_net", TARGET}.issubset(chunk.columns):
            mv = pd.to_numeric(chunk["market_value"], errors="coerce")
            lp = pd.to_numeric(chunk["listprice_with_equipment_net"], errors="coerce")
            yt = pd.to_numeric(chunk[TARGET], errors="coerce")
            ok = mv.notna() & lp.notna() & yt.notna() & (lp > 0)
            if ok.any():
                ratio = 100.0 * (mv[ok] / lp[ok])
                diff = (ratio - yt[ok]).abs()
                max_abs_diff_ratio = max(max_abs_diff_ratio, float(diff.max()))
                n_ratio_checked += int(ok.sum())

    # ---- report ----
    print(f"Path: {DATA_PATH}")
    print(f"Rows: {n_rows:,}")
    print(f"Columns ({len([c for c in pd.read_csv(DATA_PATH, nrows=0).columns if c!='Unnamed: 0'])}):")
    cols = [c for c in pd.read_csv(DATA_PATH, nrows=0).columns if c != "Unnamed: 0"]
    print(cols)

    print("\nMissingness (top 20 by count):")
    miss_items = sorted(((c, int(missing[c])) for c in cols), key=lambda x: x[1], reverse=True)
    for c, m in miss_items[:20]:
        print(f"  {c:28} missing={m:10,}  ({m/max(n_rows,1):.2%})")

    print("\nNumeric summary (computed via chunked coercion):")
    for c, st in sorted(num_stats.items()):
        n = st["n"]
        mean = st["sum"] / n if n else float("nan")
        var = (st["sum2"] / n - mean * mean) if n else float("nan")
        std = math.sqrt(var) if var == var and var >= 0 else float("nan")
        print(f"  {c:28} n={n:10,}  mean={mean:10.4f}  std={std:10.4f}  min={st['min']:.4f}  max={st['max']:.4f}")

    print("\nCategorical columns:")
    for c in CAT_COLS:
        if c not in cols:
            continue
        uniq = len(cat_uniques[c])
        uniq_str = f">={UNIQUE_CAP}" if cat_capped[c] else str(uniq)
        print(f"  {c:22} uniques(sample-capped)={uniq_str}")
        for v, k in cat_top[c].most_common(TOPK):
            print(f"    {v:24} {k:10,}")

    if n_ratio_checked > 0:
        print(f"\nCheck: y_true vs 100*market_value/listprice (checked {n_ratio_checked:,} rows)")
        print(f"  max_abs_diff = {max_abs_diff_ratio:.6f} (should be ~0 if exactly derived)")

except Exception as e:
    # fallback: minimal header + first few rows without pandas
    import csv
    print("Pandas path failed, using csv fallback:", repr(e))
    with DATA_PATH.open(newline="", encoding="utf-8", errors="replace") as f:
        r = csv.reader(f)
        header = next(r)
        print("Columns:", header)
        for i in range(5):
            print(next(r))


Path: /home/canelo/uncertainty_quantification/datasets/rv_dataset_2024.csv
Rows: 401,450
Columns (14):
['id', 'registration_year', 'sales_year', 'mbv_code', 'mbv_model', 'vda_make', 'vda_model', 'fuel', 'mileage', 'age', 'listprice_with_equipment_net', 'market_value', 'latest_sales_quarter', 'y_true']

Missingness (top 20 by count):
  id                           missing=         0  (0.00%)
  registration_year            missing=         0  (0.00%)
  sales_year                   missing=         0  (0.00%)
  mbv_code                     missing=         0  (0.00%)
  mbv_model                    missing=         0  (0.00%)
  vda_make                     missing=         0  (0.00%)
  vda_model                    missing=         0  (0.00%)
  fuel                         missing=         0  (0.00%)
  mileage                      missing=         0  (0.00%)
  age                          missing=         0  (0.00%)
  listprice_with_equipment_net missing=         0  (0.00%)
  market_value  

In [2]:
import pandas as pd
import numpy as np

path = "/home/canelo/uncertainty_quantification/datasets/rv_dataset_2024.csv"
df = pd.read_csv(path)

TARGET = "y_true"
NUM = ["registration_year","mileage","age","listprice_with_equipment_net","market_value"]
CAT = ["vda_make","vda_model","fuel","latest_sales_quarter","mbv_model","mbv_code"]

print("rows", len(df), "cols", len(df.columns))
print("id unique?", df["id"].is_unique)

# Sanity: target derivation + leakage warning
ratio = 100.0 * df["market_value"] / df["listprice_with_equipment_net"]
print("max abs(y_true - 100*mv/lp):", float((df[TARGET] - ratio).abs().max()))
print("\nWARNING: if TARGET=y_true, do NOT use market_value as a feature.")

# Numeric correlations to target (quick signal)
corr = df[NUM + [TARGET]].corr(numeric_only=True)[TARGET].sort_values(ascending=False)
print("\nCorr with y_true:\n", corr)

# Time/quarter shift in target distribution
g = df.groupby("latest_sales_quarter")[TARGET]
summary = g.agg(["count","mean","std","min","max"]).sort_index()
q = g.quantile([0.05,0.5,0.95]).unstack()
print("\nTarget by latest_sales_quarter:\n", summary.join(q.rename(columns={0.05:"p05",0.5:"p50",0.95:"p95"})))

# Cardinalities (for encoding choices)
print("\nCardinalities:")
for c in CAT:
    if c in df.columns:
        print(f"  {c:20} nunique={df[c].nunique()}")


rows 401450 cols 15
id unique? True
max abs(y_true - 100*mv/lp): 6.793456364562189e-06


Corr with y_true:
 y_true                          1.000000
registration_year               0.338366
market_value                    0.142962
age                            -0.358240
listprice_with_equipment_net   -0.368904
mileage                        -0.555897
Name: y_true, dtype: float64

Target by latest_sales_quarter:
                        count       mean       std        min        max  \
latest_sales_quarter                                                      
Y2023Q4                66778  51.261991  9.604515  15.658026  88.064285   
Y2024Q1               121334  51.624458  9.661092  15.658026  90.048859   
Y2024Q2               112320  51.444914  9.466335  19.652666  84.114349   
Y2024Q3               101018  51.052070  9.193135  18.680231  85.063843   

                            p05        p50        p95  
latest_sales_quarter                                   
Y2023Q4             